# Predicción de Precios de Viviendas (Ames Housing Dataset)
### Evaluación Parcial N°1 — MLY1101 Machine Learning
**Caso B: Predicción de precios de viviendas (Housing Prices)**

---

**Metodología:** CRISP-DM (Cross Industry Standard Process for Data Mining)

**Autor(es):** _[Nicolas Carrasco, Martin Corvalan, Martin Naranjo]_

**Fecha:** _[10-09-2026]_

---

## Índice
1. [Descripción del contexto / problema de negocio](#1)
2. [Objetivos del proyecto](#2)
3. [Definición inicial de criterio de salida para el/los algoritmo(s) de ML](#3)
4. [Selección inicial de algoritmo (máximo 3)](#4)
5. [Determinación y justificación de las métricas seleccionadas](#5)
6. [Criterio de Salida: umbral de métricas para considerar el modelo válido](#6)
7. [Descripción de las fuentes de datos utilizadas](#7)
8. [Análisis Exploratorio de los Datos (EDA)](#8)
9. [Preparación inicial de los Datos](#9)
10. [Análisis de Sesgos](#10)
11. [Consideraciones de Ética y Privacidad](#11)


<a id="1"></a>
## 1. Descripción del contexto / problema de negocio

Una inmobiliaria / portal de tasación necesita **estimar de forma automática y objetiva el precio de venta de una vivienda** a partir de sus características físicas, de ubicación y de calidad constructiva. Actualmente la tasación depende en gran medida del criterio de un agente o tasador, lo que genera:

- **Inconsistencia** entre tasaciones de propiedades similares.
- **Tiempos de respuesta lentos** para dar un precio de referencia a un cliente.
- **Dificultad para justificar** de manera cuantitativa el precio sugerido (qué atributos suben o bajan el valor).

El dataset utilizado es **Ames Housing Dataset** (Ames, Iowa, EE.UU.), que contiene **2.930 registros de ventas residenciales** ocurridas entre 2006 y 2010, con **82 variables** que describen aspectos como superficie, calidad de materiales, año de construcción, garage, sótano, barrio (`Neighborhood`), condiciones de venta, entre otras.


<a id="2"></a>
## 2. Objetivos del proyecto

**Objetivo general:**
> Desarrollar un modelo de Machine Learning de regresión capaz de predecir el precio de venta (`SalePrice`) de una vivienda en Ames, Iowa, a partir de sus características estructurales, de calidad y de ubicación.

**Objetivos específicos:**
1. Comprender la estructura, calidad y distribución de los datos disponibles (EDA).
2. Identificar las variables con mayor poder explicativo sobre el precio de venta.
3. Preparar y transformar los datos (limpieza, imputación, codificación, escalado) dejándolos listos para el proceso de modelamiento.
4. Evaluar posibles sesgos, aspectos éticos y de privacidad asociados al uso de estos datos.
5. Dejar una base reproducible (notebook + datos + informe) que permita, en una etapa posterior, entrenar y comparar modelos de predicción.


<a id="3"></a>
## 3. Definición inicial de criterio de salida para el/los algoritmo(s) de Machine Learning

Antes de seleccionar un algoritmo específico, se debe definir **qué tipo de problema de ML es este y qué se entenderá por "éxito"** del modelo:

- **Tipo de problema:** `SalePrice` es una variable **numérica continua**, por lo tanto este es un problema de **aprendizaje supervisado de regresión** (no de clasificación ni de clustering).
- **Criterio de salida general:** el modelo se considerará una solución viable si es capaz de **predecir el precio de venta con un error lo suficientemente bajo y consistente** como para servir de referencia confiable a un tasador o agente inmobiliario, y si dicho error **no varía de forma desproporcionada entre distintos segmentos** (por ejemplo, entre barrios de distinto nivel socioeconómico o entre viviendas de bajo y alto valor).
- En términos operativos, esto se traduce en exigir un **buen ajuste global** (medido con R²) combinado con un **error absoluto/porcentual acotado** (MAE, RMSE, MAPE), evaluados siempre sobre datos de prueba no usados en el entrenamiento, de modo de asegurar que el modelo generalice y no esté sobreajustado (*overfitting*).
- Este criterio de salida general se formaliza de manera cuantitativa en la sección 6 (Criterio de salida: umbral de métricas), una vez seleccionadas las métricas específicas en la sección 5.


<a id="4"></a>
## 4. Selección inicial de algoritmo (máximo 3)

Para un problema de regresión tabular como este, con una mezcla de variables numéricas y categóricas y ~2.900 registros, se preseleccionan los siguientes **3 algoritmos candidatos**, que serán entrenados y comparados en la etapa de modelamiento (fuera del alcance de esta evaluación):

| N° | Algoritmo | Justificación de la selección |
|---|---|---|
| 1 | **Regresión Lineal Múltiple** (con regularización Ridge/Lasso) | Modelo **base (baseline)** simple e interpretable. Permite cuantificar de forma directa el efecto de cada variable sobre el precio y sirve como punto de comparación mínimo exigible para los modelos más complejos. Funciona especialmente bien tras la transformación logarítmica de `SalePrice`, que linealiza parcialmente su relación con las variables predictoras. |
| 2 | **Random Forest Regressor** | Modelo de **ensamble basado en árboles** capaz de capturar relaciones no lineales e interacciones entre variables (ej. entre `Overall Qual` y `Neighborhood`) sin necesidad de especificarlas manualmente. Es robusto frente a outliers y variables poco relevantes, y entrega una medida de **importancia de variables** útil para la interpretabilidad del negocio. |
| 3 | **Gradient Boosting Regressor** (ej. XGBoost / LightGBM) | Suele ofrecer el **mejor desempeño predictivo** en datasets tabulares de tamaño medio como este, al construir árboles de forma secuencial que corrigen los errores de los anteriores. Se incluye como candidato de "techo de desempeño" (*performance ceiling*), a costa de una menor interpretabilidad directa que la regresión lineal. |

La comparación entre estos tres modelos permitirá evaluar el **trade-off entre interpretabilidad y desempeño predictivo**, algo especialmente relevante en un caso de uso donde el negocio (agentes/tasadores) necesita poder explicar el precio sugerido a sus clientes.


<a id="5"></a>
## 5. Determinación y justificación de las métricas seleccionadas

Dado que se trata de un problema de **regresión**, se descartan métricas de clasificación (accuracy, precision/recall, F1, AUC) y se seleccionan las siguientes métricas, cada una aportando una perspectiva distinta y complementaria del error del modelo:

| Métrica | Fórmula / Descripción | Por qué se selecciona |
|---|---|---|
| **RMSE** (Root Mean Squared Error) | Raíz del promedio de los errores al cuadrado | Está en las mismas unidades que `SalePrice` (dólares), es fácilmente interpretable para el negocio, y **penaliza más fuertemente los errores grandes**, relevante porque un error grande en la tasación de una vivienda de alto valor tiene un impacto económico significativo. |
| **MAE** (Mean Absolute Error) | Promedio del valor absoluto de los errores | Complementa al RMSE al ser **menos sensible a outliers**; representa el "error típico" esperado en dólares y es muy fácil de comunicar a un usuario de negocio ("en promedio nos equivocamos en US$ X"). |
| **R²** (Coeficiente de determinación) | Proporción de la varianza de `SalePrice` explicada por el modelo | Permite evaluar el **ajuste global** del modelo de forma estandarizada (0 a 1), independiente de la escala de precios, facilitando comparar el desempeño entre los 3 algoritmos candidatos. |
| **MAPE** (Mean Absolute Percentage Error) | Error absoluto promedio expresado como porcentaje del valor real | Es la métrica más intuitiva para el negocio inmobiliario, que suele hablar en términos porcentuales ("el modelo se equivoca en promedio un X% del valor de la vivienda"), y permite comparar el error relativo entre viviendas baratas y caras. |

No se utilizan métricas como accuracy o F1-score porque **no aplican a un problema de regresión**; tampoco se prioriza únicamente el R² porque, por sí solo, no informa la magnitud del error en términos monetarios, que es lo que finalmente le importa al negocio.


<a id="6"></a>
## 6. Criterio de Salida: umbral de métricas seleccionadas para considerar el modelo desarrollado como válido

En base a las métricas definidas en la sección 5, se establece el siguiente **umbral mínimo de aceptación** que deberá alcanzar el mejor modelo (evaluado sobre el conjunto de prueba, nunca sobre datos de entrenamiento) para considerarse una solución válida para el negocio:

| Métrica | Umbral de aceptación | Interpretación de negocio |
|---|---|---|
| **R²** | ≥ 0.80 | El modelo explica al menos el 80% de la variabilidad del precio de venta |
| **RMSE** | ≤ US$ 30.000 | El error cuadrático promedio no debe superar los US$ 30.000, aprox. un 17-19% del precio mediano (US$ 160.000) |
| **MAE** | ≤ US$ 20.000 | En promedio, el modelo no debe equivocarse en más de US$ 20.000 |
| **MAPE** | ≤ 12% | El error porcentual promedio no debe superar el 12% del valor real de la vivienda |

Si el mejor modelo entrenado **no alcanza estos umbrales**, se considerará necesario volver a etapas anteriores del proceso CRISP-DM (por ejemplo, ingeniería de variables adicional, prueba de otros algoritmos o recolección de más datos) antes de dar por válida la solución. Estos umbrales son una referencia inicial, definida con criterio de negocio, y podrán ajustarse en conjunto con los interesados (*stakeholders*) una vez obtenidos los primeros resultados de modelamiento.


<a id="7"></a>
## 7. Descripción de las fuentes de datos utilizadas

- **Fuente de datos:** `Ames_Iowa_Housing_Dataset.csv`, dataset público de viviendas vendidas en la ciudad de Ames, Iowa (EE.UU.) entre 2006 y 2010, compilado originalmente por Dean De Cock (2011) como alternativa moderna al clásico dataset de Boston Housing. Es ampliamente utilizado en la comunidad de ciencia de datos (ej. competencia *House Prices* de Kaggle).
- **Formato:** archivo CSV, 2.930 filas x 82 columnas.
- **Variable objetivo:** `SalePrice` (precio de venta en dólares).
- **Tipos de variables:** mezcla de variables numéricas (superficies, año de construcción, cantidad de baños, etc.) y categóricas (calidad de materiales, tipo de barrio, tipo de garage, etc.), además de dos columnas identificadoras (`Order`, `PID`) sin valor predictivo.
- **Herramientas y librerías utilizadas:**
  - **Python 3** como lenguaje principal.
  - **pandas / numpy** para manipulación y análisis de datos.
  - **matplotlib / seaborn** para visualización.
  - **scikit-learn** para preprocesamiento (imputación, encoding, escalado) y, en una etapa posterior, modelamiento.
  - **Jupyter Notebook** como entorno de trabajo reproducible.
  - **Git / GitHub** (o una carpeta compartida en Google Drive) como herramienta colaborativa para versionar el proyecto entre los integrantes del equipo, permitiendo trabajar en paralelo sobre el mismo notebook y dataset sin sobrescribir el trabajo de los demás.

A continuación se importan las librerías necesarias y se realiza la carga inicial del dataset.


In [1]:
# Librerías de manipulación de datos
import pandas as pd
import numpy as np

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de preprocesamiento (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 100)

import warnings
warnings.filterwarnings("ignore")

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


In [2]:
# Carga del dataset desde la carpeta data
df = pd.read_csv("../data/Ames_Iowa_Housing_Dataset.csv")

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/Ames_Iowa_Housing_Dataset.csv'

In [ ]:
# Tipos de datos y cantidad de valores no nulos por columna
df.info()


In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe().T


In [ ]:
# Estadísticas descriptivas de variables categóricas
df.describe(include="object").T


**Observaciones iniciales:**
- El dataset combina **variables numéricas** con **variables categóricas**, como se anticipó.
- La variable objetivo es **`SalePrice`**.
- Las columnas `Order` y `PID` son identificadoras y se excluirán del proceso de modelamiento.


<a id="8"></a>
## 8. Análisis Exploratorio de los Datos (EDA)

En esta sección se analiza la **calidad de los datos** (valores nulos, duplicados, outliers) y se identifican **patrones y variables relevantes** relacionadas con el precio de venta.

### 8.1 Calidad de los datos: valores duplicados


In [ ]:
# Verificación de filas duplicadas y de identificadores únicos (PID)
print(f"Filas totalmente duplicadas: {df.duplicated().sum()}")
print(f"Identificadores de propiedad (PID) duplicados: {df['PID'].duplicated().sum()}")


No se detectan filas duplicadas ni identificadores de propiedad repetidos, lo que indica que cada registro corresponde a una venta única.

### 8.2 Calidad de los datos: valores nulos (missing values)


In [ ]:
# Cálculo de valores nulos por columna (cantidad y porcentaje)
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)

tabla_nulos = pd.DataFrame({
    "n_nulos": nulos,
    "pct_nulos": nulos_pct
})
tabla_nulos = tabla_nulos[tabla_nulos["n_nulos"] > 0].sort_values("pct_nulos", ascending=False)
tabla_nulos


In [ ]:
# Visualización de las variables con mayor porcentaje de valores nulos
plt.figure(figsize=(10, 8))
sns.barplot(x=tabla_nulos["pct_nulos"], y=tabla_nulos.index, color="steelblue")
plt.xlabel("Porcentaje de valores nulos (%)")
plt.ylabel("Variable")
plt.title("Porcentaje de valores nulos por variable")
plt.tight_layout()
plt.savefig("../images/valores_nulos.png", dpi=120)
plt.show()


**Interpretación de los valores nulos:**

Es fundamental notar que, según el diccionario de datos del Ames Housing Dataset, **la mayoría de estos "nulos" no son datos faltantes por error, sino que representan la ausencia física del atributo**. Por ejemplo:

- `Pool QC` nulo → la vivienda **no tiene piscina**.
- `Alley` nulo → la vivienda **no tiene acceso por callejón**.
- `Fence` nulo → la vivienda **no tiene cerco**.
- `Fireplace Qu` nulo → la vivienda **no tiene chimenea**.
- `Garage Type/Finish/Qual/Cond/Yr Blt` nulos → la vivienda **no tiene garage**.
- `Bsmt Qual/Cond/Exposure/BsmtFin Type 1-2` nulos → la vivienda **no tiene sótano**.

Solo un pequeño grupo de variables corresponde a **verdaderos datos faltantes** que deberán imputarse en la etapa de preparación:
- `Lot Frontage` (frontis del lote, ~16,7% de nulos): probablemente no se registró en terreno.
- `Mas Vnr Type` / `Mas Vnr Area` (revestimiento de mampostería): pocos casos, posible omisión de registro.
- `Electrical`, `Garage Cars`, `Garage Area`, `Total Bsmt SF`, `Bsmt Full/Half Bath`, `BsmtFin SF 1/2`, `Bsmt Unf SF`: 1 o 2 casos aislados, probablemente error de captura.

Esta distinción es clave: para las variables de "ausencia de atributo" se imputará el valor `"None"` (categóricas) o `0` (numéricas), mientras que para los verdaderos faltantes se usará una imputación estadística (mediana / moda). Esto se ejecuta en la sección 9.

### 8.3 Análisis de la variable objetivo: `SalePrice`


In [ ]:
print(df["SalePrice"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribución de SalePrice")
axes[0].set_xlabel("Precio de venta (US$)")

sns.boxplot(x=df["SalePrice"], ax=axes[1], color="lightcoral")
axes[1].set_title("Boxplot de SalePrice (detección de outliers)")
axes[1].set_xlabel("Precio de venta (US$)")

plt.tight_layout()
plt.savefig("../images/distribucion_saleprice.png", dpi=120)
plt.show()


**Interpretación:**
- La variable `SalePrice` presenta una **distribución asimétrica hacia la derecha (right-skewed)**: la mayoría de las viviendas se concentran entre US$ 130.000 y US$ 215.000 (percentiles 25 y 75), pero existen algunas viviendas de muy alto valor (hasta US$ 755.000) que estiran la distribución.
- Esta asimetría sugiere que, en la etapa de modelamiento, podría ser beneficioso aplicar una **transformación logarítmica** (`log(SalePrice)`) para normalizar la distribución y mejorar el desempeño de modelos como la Regresión Lineal (candidato N°1 de la sección 4).
- El boxplot confirma la presencia de **outliers en el extremo superior** (viviendas de lujo), que deberán analizarse con cuidado: no necesariamente son errores, pueden ser casos legítimos que requieren tratamiento especial.

### 8.4 Relación entre variables numéricas y el precio de venta


In [ ]:
# Matriz de correlación de variables numéricas con SalePrice
num_cols = df.select_dtypes(include=[np.number]).columns.drop(["Order", "PID"])
correlaciones = df[num_cols].corr()["SalePrice"].sort_values(ascending=False)
correlaciones


In [ ]:
# Top 10 variables más correlacionadas (positiva o negativamente) con SalePrice
top_corr = correlaciones.drop("SalePrice").abs().sort_values(ascending=False).head(10).index
top_corr_valores = correlaciones[top_corr]

plt.figure(figsize=(9, 6))
sns.barplot(x=top_corr_valores.values, y=top_corr_valores.index, palette="viridis")
plt.title("Top 10 variables numéricas más correlacionadas con SalePrice")
plt.xlabel("Coeficiente de correlación de Pearson")
plt.tight_layout()
plt.savefig("../images/top_correlaciones.png", dpi=120)
plt.show()


In [ ]:
# Mapa de calor de correlaciones entre las variables más relevantes
top_vars = list(top_corr) + ["SalePrice"]
plt.figure(figsize=(10, 8))
sns.heatmap(df[top_vars].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Mapa de calor: correlación entre variables clave y SalePrice")
plt.tight_layout()
plt.savefig("../images/heatmap_correlaciones.png", dpi=120)
plt.show()


**Patrones identificados:**
- `Overall Qual` (calidad general de materiales y acabados) es, por lejos, la variable con **mayor correlación positiva** con el precio de venta.
- `Gr Liv Area` (superficie habitable sobre el nivel del suelo) y `Garage Cars`/`Garage Area` también muestran correlaciones altas: a mayor superficie y capacidad de garage, mayor precio.
- `Total Bsmt SF` y `1st Flr SF` (superficie del sótano y del primer piso) refuerzan la idea de que **el tamaño construido es un factor central** en el precio.
- `Year Built` y `Year Remod/Add` muestran que las viviendas más nuevas o remodeladas recientemente tienden a tener mayor valor.

### 8.5 Relación entre variables categóricas relevantes y el precio de venta


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

# Precio promedio por barrio (Neighborhood)
orden_barrios = df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=orden_barrios, ax=axes[0])
axes[0].set_title("Distribución de SalePrice por Barrio (Neighborhood)")
axes[0].tick_params(axis="x", rotation=90)

# Precio promedio por calidad general (Overall Qual)
sns.boxplot(data=df, x="Overall Qual", y="SalePrice", ax=axes[1], color="lightgreen")
axes[1].set_title("Distribución de SalePrice por Calidad General (Overall Qual)")

plt.tight_layout()
plt.savefig("../images/saleprice_por_categoria.png", dpi=120)
plt.show()


**Interpretación:**
- Existe una **variabilidad considerable en el precio según el barrio** (`Neighborhood`): barrios como *NridgHt*, *NoRidge* o *StoneBr* concentran las viviendas más caras, mientras que otros como *MeadowV* o *IDOTRR* concentran las más económicas. Esto confirma que la **ubicación es una variable clave** a incluir en el modelo (este hallazgo se retoma en la sección 10, Análisis de Sesgos).
- Se observa una **relación monotónica clara** entre `Overall Qual` (escala 1 a 10) y `SalePrice`: a mayor calidad, mayor precio, sin excepciones relevantes.

### 8.6 Detección de outliers (variables numéricas clave)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x="Gr Liv Area", y="SalePrice", ax=axes[0], alpha=0.5)
axes[0].set_title("SalePrice vs. Superficie Habitable (Gr Liv Area)")

sns.scatterplot(data=df, x="Total Bsmt SF", y="SalePrice", ax=axes[1], alpha=0.5, color="orange")
axes[1].set_title("SalePrice vs. Superficie de Sótano (Total Bsmt SF)")

plt.tight_layout()
plt.savefig("../images/outliers_scatter.png", dpi=120)
plt.show()

# Identificación explícita de outliers evidentes: viviendas muy grandes pero muy baratas
outliers_evidentes = df[(df["Gr Liv Area"] > 4000) & (df["SalePrice"] < 300000)]
print(f"Outliers evidentes detectados (Gr Liv Area > 4000 y SalePrice < 300,000): {len(outliers_evidentes)}")
outliers_evidentes[["PID", "Gr Liv Area", "SalePrice", "Neighborhood", "Sale Condition"]]


**Interpretación:** se identifican un par de viviendas con **superficie habitable muy grande (> 4.000 pies²) pero precio de venta bajo**, un patrón atípico que rompe la relación esperada tamaño-precio. Estos casos suelen corresponder a ventas con condiciones especiales (`Sale Condition` distinto de `Normal`, por ejemplo ventas entre familiares o liquidaciones) y son candidatos a **eliminarse o tratarse por separado** en la etapa de modelamiento, ya que podrían distorsionar el ajuste del modelo.

### 8.7 Resumen de hallazgos del EDA

| Hallazgo | Implicancia para la preparación de datos |
|---|---|
| Distribución asimétrica de `SalePrice` | Evaluar transformación logarítmica |
| Nulos "estructurales" (Pool, Garage, Bsmt, etc.) | Imputar como `"None"` / `0`, no eliminar filas |
| Nulos reales aislados (`Lot Frontage`, `Electrical`, etc.) | Imputar con mediana (numéricas) / moda (categóricas) |
| `Overall Qual`, `Gr Liv Area`, `Garage Cars/Area` fuertemente correlacionados con el precio | Priorizar como variables predictoras |
| `Neighborhood` con fuerte efecto sobre el precio | Codificar (One-Hot Encoding) e incluir en el modelo |
| Outliers de viviendas grandes y baratas | Evaluar exclusión o tratamiento robusto en modelamiento |
| Columnas `Order` y `PID` son identificadores | Excluir del set de variables predictoras |


<a id="9"></a>
## 9. Preparación inicial de los Datos

En esta etapa se transforma el dataset crudo en un conjunto de datos limpio y estructurado, listo para el proceso de modelamiento (que se desarrollará en una etapa posterior del proyecto).

### 9.1 Eliminación de columnas identificadoras y outliers evidentes


In [ ]:
# Copia de trabajo para no modificar el dataset original
df_prep = df.copy()

# Eliminación de identificadores sin valor predictivo
df_prep = df_prep.drop(columns=["Order", "PID"])

# Eliminación de outliers evidentes detectados en el EDA
antes = df_prep.shape[0]
df_prep = df_prep[~((df_prep["Gr Liv Area"] > 4000) & (df_prep["SalePrice"] < 300000))]
despues = df_prep.shape[0]
print(f"Filas eliminadas por outliers evidentes: {antes - despues}")
print(f"Dimensiones tras limpieza inicial: {df_prep.shape}")


### 9.2 Tratamiento de valores nulos "estructurales" (ausencia de atributo)

Se imputa `"None"` para variables categóricas y `0` para variables numéricas cuando el nulo representa la **ausencia física del atributo** (no un dato faltante real).


In [ ]:
# Variables categóricas donde NaN significa "no tiene el atributo"
cols_none_cat = [
    "Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
    "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
    "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2",
    "Mas Vnr Type"
]
for col in cols_none_cat:
    df_prep[col] = df_prep[col].fillna("None")

# Variables numéricas donde NaN significa "0" (no tiene el atributo)
cols_none_num = [
    "Garage Yr Blt", "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2",
    "Bsmt Unf SF", "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath",
    "Garage Cars", "Garage Area"
]
for col in cols_none_num:
    df_prep[col] = df_prep[col].fillna(0)

print("Valores nulos estructurales imputados.")


### 9.3 Tratamiento de valores nulos reales (imputación estadística)

Para las variables restantes con nulos (`Lot Frontage` y `Electrical`), se aplica imputación estadística: **mediana** para numéricas y **moda** para categóricas, ya que representan datos que probablemente no fueron registrados en terreno y no una ausencia de atributo.


In [ ]:
# Verificación de nulos restantes antes de la imputación estadística
nulos_restantes = df_prep.isnull().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]
print("Nulos restantes antes de imputación estadística:")
print(nulos_restantes)


In [ ]:
# Imputación de Lot Frontage con la mediana (agrupada por barrio, ya que el frontis
# del lote depende fuertemente de la ubicación / tipo de barrio)
df_prep["Lot Frontage"] = df_prep.groupby("Neighborhood")["Lot Frontage"].transform(
    lambda x: x.fillna(x.median())
)

# Imputación de Electrical con la moda (valor más frecuente)
df_prep["Electrical"] = df_prep["Electrical"].fillna(df_prep["Electrical"].mode()[0])

# Verificación final de valores nulos
print(f"Total de valores nulos restantes en el dataset: {df_prep.isnull().sum().sum()}")


### 9.4 Transformación de la variable objetivo

Dado que el EDA (sección 8.3) mostró una distribución asimétrica de `SalePrice`, se crea una versión transformada `SalePrice_log` mediante logaritmo natural, útil para futuros modelos lineales (ver sección 4).


In [ ]:
df_prep["SalePrice_log"] = np.log1p(df_prep["SalePrice"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df_prep["SalePrice"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("SalePrice original")
sns.histplot(df_prep["SalePrice_log"], kde=True, ax=axes[1], color="seagreen")
axes[1].set_title("SalePrice transformado (log)")
plt.tight_layout()
plt.savefig("../images/transformacion_log_saleprice.png", dpi=120)
plt.show()


La transformación logarítmica reduce notoriamente la asimetría de la variable objetivo, acercándola a una distribución normal, lo que favorecerá el desempeño de modelos como la Regresión Lineal en etapas posteriores.

### 9.5 Codificación de variables categóricas

Se separan las variables predictoras (`X`) del objetivo (`y`), y se identifican las columnas numéricas y categóricas para aplicar la codificación adecuada.


In [ ]:
# Separación de variables predictoras y variable objetivo
X = df_prep.drop(columns=["SalePrice", "SalePrice_log"])
y = df_prep["SalePrice_log"]  # se usa la versión log-transformada como objetivo de modelamiento

# Identificación de columnas numéricas y categóricas
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Variables numéricas: {len(num_features)}")
print(f"Variables categóricas: {len(cat_features)}")


### 9.6 Pipeline de preprocesamiento (imputación + escalado + codificación)

Se construye un `Pipeline` de `scikit-learn` que:
1. Imputa valores numéricos faltantes con la mediana (medida de seguridad adicional).
2. Escala las variables numéricas con `StandardScaler`.
3. Codifica variables categóricas con `OneHotEncoder`.

Este pipeline permite aplicar exactamente las mismas transformaciones al set de entrenamiento y de prueba, evitando fuga de información (*data leakage*).


In [ ]:
# Pipeline para variables numéricas
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline para variables categóricas
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combinación de ambos pipelines en un ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

preprocessor


### 9.7 División en conjuntos de entrenamiento y prueba

Se reserva un **80%** de los datos para entrenamiento y un **20%** para prueba. Esta división es la que se usará en la etapa de modelamiento para evaluar los 3 algoritmos candidatos (sección 4) según las métricas y umbrales definidos (secciones 5 y 6).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Set de entrenamiento: {X_train.shape[0]} filas")
print(f"Set de prueba: {X_test.shape[0]} filas")


In [ ]:
# Ajuste del preprocesador SOLO con datos de entrenamiento (para evitar data leakage)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Dimensiones de X_train luego del preprocesamiento: {X_train_transformed.shape}")
print(f"Dimensiones de X_test luego del preprocesamiento: {X_test_transformed.shape}")


### 9.8 Guardado de los datos preparados

Se exportan los conjuntos de datos limpios y transformados a la carpeta `data/`, dejando el proyecto listo para la siguiente etapa de modelamiento.


In [ ]:
import joblib

# Dataset limpio (sin transformaciones de encoding, útil para futuras exploraciones)
df_prep.to_csv("../data/housing_clean.csv", index=False)

# División de datos lista para modelamiento
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

# Se guarda el pipeline de preprocesamiento entrenado para reutilizarlo en el modelamiento
joblib.dump(preprocessor, "../models/preprocessor_pipeline.pkl")

print("Archivos de datos preparados y pipeline de preprocesamiento guardados exitosamente.")


<a id="10"></a>
## 10. Análisis de Sesgos

- **Sesgo geográfico/temporal:** el dataset corresponde únicamente a la ciudad de **Ames, Iowa**, entre **2006 y 2010** (periodo que incluye la crisis subprime en EE.UU.). Un modelo entrenado con estos datos **no debería generalizarse** a otras ciudades, países, ni a condiciones de mercado muy distintas (por ejemplo, el mercado inmobiliario chileno) sin una validación adicional.
- **Sesgo socioeconómico vía `Neighborhood`:** como se observó en la sección 8.5, el barrio es una variable con alto poder predictivo. Esto implica un riesgo de que el modelo **replique y refuerce desigualdades históricas** entre barrios (por ejemplo, barrios históricamente menos valorados seguirán prediciéndose con precios bajos, incluso si las condiciones cambian). Es importante que el modelo se use como **apoyo a la decisión** y no como única fuente de verdad, especialmente en contextos donde podría influir en decisiones de crédito hipotecario o políticas urbanas.
- **Sesgo de selección:** el dataset solo incluye **viviendas efectivamente vendidas**, por lo que no representa a la totalidad del parque habitacional (quedan fuera viviendas que no se transaron en el periodo, lo cual podría subrepresentar ciertos segmentos del mercado).
- **Variables sensibles indirectas (proxy bias):** si bien no hay variables explícitas de raza, género o nivel socioeconómico de las personas, variables como `Neighborhood` pueden actuar como **proxy** de composición socioeconómica o étnica de un sector, un fenómeno documentado en la literatura de *fairness in ML* para modelos de tasación inmobiliaria. Se recomienda, en etapas futuras de modelamiento, evaluar el desempeño del modelo de forma **desagregada por barrio** para detectar posibles inequidades sistemáticas en el error de predicción (por ejemplo, verificar si el MAE/MAPE es sistemáticamente peor en ciertos barrios).
- **Mitigación propuesta:** documentar explícitamente el alcance geográfico/temporal del modelo, monitorear métricas de error por subgrupo (`Neighborhood`) y evitar el uso del modelo como única fuente de decisión en contextos con impacto directo sobre personas (créditos, seguros).


<a id="11"></a>
## 11. Consideraciones de Ética y Privacidad

### 11.1 Privacidad de los datos

- El dataset **no contiene información personal identificable (PII)**: no incluye nombres de propietarios, direcciones exactas, RUT/SSN, ni datos de contacto. El identificador de cada registro (`PID`) corresponde a un número de parcela catastral (*Parcel ID*), de carácter público en Ames, Iowa, y no permite identificar directamente a una persona natural.
- Aun así, se recomienda **no cruzar este dataset con otras fuentes** (por ejemplo, registros públicos de propietarios) que sí podrían re-identificar a personas, ya que la combinación de variables muy específicas (ubicación exacta + características únicas de la vivienda) podría en teoría acotar la identificación de un inmueble específico y, por extensión, de su propietario.

### 11.2 Consideraciones éticas de uso

- El modelo resultante debe entenderse como una **herramienta de apoyo a la tasación**, no como un reemplazo del criterio profesional de un tasador certificado, especialmente en decisiones con impacto financiero relevante (créditos hipotecarios, seguros, etc.).
- Se recomienda **transparentar las variables** que más influyen en cada predicción (por ejemplo, mediante técnicas de interpretabilidad como SHAP en etapas futuras), de modo que un cliente pueda entender por qué se le entrega determinado precio de referencia.
- Cualquier despliegue productivo del modelo debería incluir un **monitoreo periódico** para detectar *data drift* (cambios en el mercado inmobiliario que vuelvan obsoletas las predicciones) y sesgos emergentes, en línea con lo identificado en la sección 10.
- El uso del modelo debe ser **consistente con la normativa local de protección de datos** vigente en el territorio donde se despliegue (por ejemplo, en Chile, la Ley N°19.628 sobre Protección de la Vida Privada), especialmente si en el futuro se integran datos adicionales que sí contengan información personal de compradores o vendedores.


## Conclusiones y próximos pasos

### Conclusiones de esta etapa

1. El dataset Ames Housing es una fuente de datos **robusta y de buena calidad** para el problema de negocio planteado, con muy pocos duplicados y valores faltantes reales (la mayoría de los "nulos" son en realidad ausencia de un atributo, no errores).
2. Las variables con mayor relación con el precio de venta son la **calidad general de la construcción** (`Overall Qual`), la **superficie habitable** (`Gr Liv Area`), la **capacidad del garage** y el **barrio** (`Neighborhood`), consistente con el sentido común inmobiliario.
3. Se aplicó una **transformación logarítmica** a la variable objetivo para corregir su asimetría, y se construyó un **pipeline reproducible** de imputación, escalado y codificación, dejando los datos listos para el modelamiento.
4. Se identificaron y trataron **outliers evidentes** (viviendas muy grandes con precios anormalmente bajos) que podrían haber distorsionado modelos futuros.
5. Se preseleccionaron **3 algoritmos candidatos** (Regresión Lineal, Random Forest, Gradient Boosting) y se definieron **métricas y umbrales concretos** (R² ≥ 0.80, RMSE ≤ US$ 30.000, MAE ≤ US$ 20.000, MAPE ≤ 12%) para evaluar objetivamente el éxito del modelo en la siguiente etapa.
6. Se documentaron los **riesgos de sesgo y las consideraciones éticas/de privacidad** más relevantes, principalmente asociados al uso de la variable `Neighborhood` como proxy socioeconómico y a la representatividad geográfica/temporal limitada del dataset.

### Próximos pasos (fuera del alcance de esta evaluación)

- Entrenar los 3 algoritmos candidatos sobre `X_train_transformed` / `y_train`.
- Evaluar cada modelo con las métricas definidas (RMSE, MAE, R², MAPE) sobre el set de prueba y contrastarlas contra los umbrales de la sección 6.
- Realizar ajuste de hiperparámetros (*hyperparameter tuning*) y validación cruzada.
- Analizar la interpretabilidad del modelo final (importancia de variables, SHAP) y su desempeño desagregado por barrio para verificar los riesgos de sesgo identificados en la sección 10.
